In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os

os.makedirs("../output", exist_ok=True)

panel = pd.read_csv("../data/stacked_event_panel.csv")


panel = panel[(panel["k"] >= -6) & (panel["k"] <= 12)].copy()
panel["k_cat"] = panel["k"].astype(str)
panel.loc[panel["k"] == -1, "k_cat"] = "ref"

model = smf.ols(
    "log_handle ~ C(k_cat, Treatment(reference='ref')) + C(state_event)",
    data=panel,
).fit(cov_type="cluster", cov_kwds={"groups": panel["state"]})

print("=" * 100)
print("STACKED EVENT-STUDY RESULTS (real data, N=%d state-event-months, %d events, %d state-event units)"
      % (len(panel), panel["event_id"].nunique(), panel["state_event"].nunique()))
print("Clustered SE by state (3 clusters: AZ, CO, CT) -- treat p-values as indicative, not exact, given so few clusters.")
print("=" * 100)

ks = sorted(panel.loc[panel["k"] != -1, "k"].unique())
rows = []
for k in ks:
    term = f"C(k_cat, Treatment(reference='ref'))[T.{k}]"
    if term in model.params.index:
        rows.append({
            "k": k,
            "beta_k": model.params[term],
            "se": model.bse[term],
            "p": model.pvalues[term],
        })
coef_table = pd.DataFrame(rows)
coef_table["sig"] = coef_table["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")
print(coef_table.to_string(index=False, formatters={
    "beta_k": "{:.3f}".format, "se": "{:.3f}".format, "p": "{:.3f}".format
}))
print(f"\nR-squared: {model.rsquared:.3f}   N obs: {int(model.nobs)}   df_resid: {int(model.df_resid)}")

coef_table.to_csv("../output/event_study_coefficients.csv", index=False)
with open("../output/event_study_model_summary.txt", "w") as f:
    f.write(str(model.summary()))

print("\nSaved: ../output/event_study_coefficients.csv, ../output/event_study_model_summary.txt")

STACKED EVENT-STUDY RESULTS (real data, N=224 state-event-months, 6 events, 12 state-event units)
Clustered SE by state (3 clusters: AZ, CO, CT) -- treat p-values as indicative, not exact, given so few clusters.
 k beta_k    se     p sig
-6 -0.178 0.139 0.201    
-5 -0.100 0.117 0.392    
-4 -0.026 0.099 0.793    
-3  0.013 0.054 0.813    
-2 -0.023 0.035 0.502    
 0  0.138 0.032 0.000 ***
 1  0.328 0.020 0.000 ***
 2  0.416 0.017 0.000 ***
 3  0.375 0.038 0.000 ***
 4  0.362 0.036 0.000 ***
 5  0.256 0.042 0.000 ***
 6  0.174 0.045 0.000 ***
 7  0.149 0.024 0.000 ***
 8  0.197 0.034 0.000 ***
 9  0.231 0.045 0.000 ***
10  0.264 0.037 0.000 ***
11  0.280 0.019 0.000 ***
12  0.373 0.022 0.000 ***

R-squared: 0.757   N obs: 224   df_resid: 194

Saved: ../output/event_study_coefficients.csv, ../output/event_study_model_summary.txt


/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 29, but rank is 2
  warnings.warn('covariance of constraints does not have full '
